# Pay for Data — Heurist Finance Agent

A finance research agent that pays for real-time market data from Heurist using AgentCore payments.

The agent calls paid endpoints via `http_request`. The `AgentCorePaymentsPlugin` intercepts HTTP 402 responses, asks the AgentCore payment manager to generate payment proofs against your payment instrument and payment session, and retries automatically.

**Before running:**
1. `pip install -r requirements.txt`
2. `cp .env.example .env` and fill in your credentials (payment manager ARN, payment session ID, payment instrument ID)
3. Run the catalog sync below

See [`README.md`](README.md) for setup details and architecture.

## Install dependencies

In [ ]:
%pip install -r requirements.txt --quiet

## Sync the Heurist tool catalog

Fetches the current registry of x402-enabled endpoints and caches it locally. The agent's system prompt is built from this catalog so it knows what URLs and parameters are available.

In [ ]:
from heurist_finance_agent.catalog import fetch_live_catalog, get_tools_for_agents
from heurist_finance_agent.config import get_config

cfg = get_config()
catalog = fetch_live_catalog()
selected = get_tools_for_agents(cfg.heurist_tool_agent_ids)

print(f"Agents in registry: {catalog['count']}")
print(f"Selected agents:    {', '.join(cfg.heurist_tool_agent_ids)}")
print(f"Loaded paid tools:  {len(selected)}")

## Run the agent

The agent is built in [`agent.py`](heurist_finance_agent/agent.py) with:
- `http_request` — calls Heurist endpoints
- `AgentCorePaymentsPlugin` — drives payment processing for HTTP 402 responses (payment manager, payment session, payment instrument)
- AgentCore Code Interpreter — sandboxed pandas/matplotlib analysis

Change the prompt below to explore different queries.

In [ ]:
from heurist_finance_agent.agent import invoke_agent

prompt = "Analyze AMZN stock performance using SEC filing data. Create a chart and a brief markdown summary."

result = invoke_agent(prompt)
print(result)

## Inspect artifacts

Charts and reports the agent produced:

In [ ]:
from pathlib import Path

artifacts_dir = Path("heurist_finance_agent/artifacts")
for path in sorted(artifacts_dir.glob("*")):
    if path.is_file():
        print(f"{path.stat().st_size:>10} bytes  {path.name}")